## Portfolio 2

In deze portfolio gaan we prompt engineering toepassen om de data van de HBO Kennisbank te webscrapen.

Daarna gaan we met deze data titels genereren op basis van de abstract van elk publicatie. Dit kunnen we vervolgens verder finetunen, samplen, RLAIF op toepassen en evalueren. 

### Prompt 1

Het doel van het eerste prompt is om context vast te leggen en alvast een stappenplan te maken voor het maken van een webscraper. Hier zeggen wij alvast wat het uiteindelijke doel is.

In deze prompt beginnen wij met het geraamte van een webscraper die specifiek werkt op wetenschappelijke publicaties van het HBO Kennisbank. Hier duiden we meteen aan wat het uiteindelijke doel is voor context en geven we meteen door welk website we willen scrapen.

Na het aanduiding van context wordt er gebruik gemaakt van een aantal prompt-engineering technieken. 
- Step-by-step prompting: expliciet gevraagd voor een stappenplan voor het uitvoeren van het webscraper.
- Beperkingen vastgezet: het wordt ook aangeduid dat het code uitvoerbaar moet zijn in Python zonder gebruik van een API.

### Antwoord 1

### Resultaat verwerken

Hoewel deze prompt code opgeleverd heeft, is het slechts een geraamte van wat gedaan moet worden om een webscraper te bouwen. Er zijn nog andere stappen die hier toegevoegd moeten worden om het daadwerkelijk te laten werken. Dat proberen wij verder uit te werken in volgende prompts. 

In [1]:
# stap 2
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup

url = "https://hbo-kennisbank.nl/searchresult?q="
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

titles = soup.find_all("h2")  # voorbeeld, moet je inspecteren

for t in titles:
    print(t.get_text(strip=True))

# stap 3
base_url = "https://hbo-kennisbank.nl/searchresult?page={}"

for page in range(1, 50):
    url = base_url.format(page)
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # parse titles

# stap 4
links = []

for item in soup.select("a.result-link"):
    links.append(item["href"])

# stap 5
def scrape_detail(url):
    r = requests.get(url, headers=headers)
    s = BeautifulSoup(r.text, "html.parser")

    title = s.find("h1").get_text(strip=True)
    abstract = s.find("div", class_="abstract")

    return {
        "title": title,
        "abstract": abstract.get_text(strip=True) if abstract else None
    }

# stap 6

data = []

for link in links:
    try:
        item = scrape_detail(link)
        data.append(item)
    except Exception as e:
        print("Error:", e)

# stap 7
time.sleep(1)

# stap 8

df = pd.DataFrame(data)
df.to_csv("hbo_kennisbank.csv", index=False)

### Prompt 2

We willen nu de stappen omzetten naar werkende code die we in een cel kunnen stoppen door een ander prompt te gebruiken. Hier is het doel om het meteen werkend te maken zonder handwerk. 

Ook voegen wij toe dat het model ook de abstract en jaar van elk publicatie kan scrapen. Dit kan handig zijn voor extra context voor het genereren van titels.

In deze prompt hebben wij een paar aandachtspunten voor het genereren van code gebruikt.

- Role prompting: Het generatieve model moet de rol van een data scientist aannemen. Dit techniek wordt gebruikt met het doel om ervoor te zorgen dat het code overzichtelijk en professioneel eruit ziet.
- Verwachtingen vastgelegd: Hierbij leggen wij vast dat het resultaat 'copy-paste klaar' moet zijn.
- Contextual chaining: het prompt bouwt voort op het stappenplan van het vorige antwoord.

### Antwoord 2

### Resultaat verwerken

Als resultaat heeft het model een 'copy-paste klaar' codecel gemaakt die we meteen kunnen gebruiken. 

Bij het uitvoeren van het code, was het merkbaar dat het lukte om alle gevraagde details in een .csv bestand te plaatsen, maar bij nader inspectie van het .csv bestand was er wel te zien dat er extra text van het pagina werd gepakt (het titel, maar ook details zoals organisatie, afdeling etc.) omdat het code voor het scrapen van een abstract de hele 'main-content' stuk pakt van het detail-pagina. Dit kunnen we in de volgende prompt aanscherpen.

Het is wel gelukt om pagina's te webscrapen, maar zonder twee kleine aanpassingen kon het webscraper alleen de eerste pagina van resultaten pakken.

Aanpassingen: 
- 'SEARCH_URL' aangepast omdat het model een aanname had gemaakt over het werking van de HBO Kennisbank. De oorspronkelijke 'SEARCH_URL' belandde altijd op de eerste pagina van resultaten, ook met een ander paginanummer.
- 'max_pages' aangepast zodat er meer pagina's worden gescrapet.

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
from urllib.parse import urljoin

BASE_URL = "https://hbo-kennisbank.nl"
SEARCH_URL = BASE_URL + "/searchresult?has-link=yes&p={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
}

# -----------------------------
# Helper: veilige tekst extractie
# -----------------------------
def safe_text(element):
    return element.get_text(strip=True) if element else None


# -----------------------------
# Stap 1: Verzamel links per pagina
# -----------------------------
def get_result_links(page):
    url = SEARCH_URL.format(page)
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        print(f"Fout bij pagina {page}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    links = []
    
    # Selector is bewust breed gehouden (site kan variëren)
    for a in soup.select("a"):
        href = a.get("href")
        if href and "/details/" in href:
            full_url = urljoin(BASE_URL, href)
            links.append(full_url)

    return list(set(links))  # duplicates verwijderen


# -----------------------------
# Stap 2: Scrape detailpagina
# -----------------------------
def scrape_detail(url):
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        # Titel
        title = safe_text(soup.find("h1"))

        # Abstract (meerdere fallback opties)
        abstract = None
        
        # optie 1
        abstract_div = soup.find("div", class_="abstract")
        if abstract_div:
            abstract = safe_text(abstract_div)

        # optie 2 fallback
        if not abstract:
            for div in soup.find_all("div"):
                if "samenvatting" in div.get_text().lower():
                    abstract = safe_text(div)
                    break

        # Jaar (meerdere strategieën)
        year = None

        # optie 1: metadata blok
        for li in soup.find_all("li"):
            text = li.get_text().lower()
            if "jaar" in text or "datum" in text:
                year = li.get_text(strip=True)
                break

        # optie 2 fallback: zoek 4-digit jaar
        if not year:
            import re
            text = soup.get_text()
            match = re.search(r"\b(20\d{2}|19\d{2})\b", text)
            if match:
                year = match.group(0)

        return {
            "title": title,
            "abstract": abstract,
            "year": year,
            "url": url
        }

    except Exception as e:
        print(f"Error bij {url}: {e}")
        return None


# -----------------------------
# Stap 3: Hoofd scraper
# -----------------------------
def scrape_hbo_kennisbank(max_pages=10, delay=1):
    all_links = []

    print("Links verzamelen...")
    for page in range(1, max_pages + 1):
        links = get_result_links(page)
        print(f"Pagina {page}: {len(links)} links")
        all_links.extend(links)
        time.sleep(delay)

    all_links = list(set(all_links))
    print(f"Totaal unieke links: {len(all_links)}")

    # Detail scraping
    data = []
    print("Detailpagina's scrapen...")

    for link in tqdm(all_links):
        item = scrape_detail(link)
        if item and item["title"]:
            data.append(item)
        time.sleep(delay)

    df = pd.DataFrame(data)
    return df


# -----------------------------
# Run scraper
# -----------------------------
if __name__ == "__main__":
    df = scrape_hbo_kennisbank(max_pages=50, delay=1)

    print(df.head())

    df.to_csv("hbo_kennisbank_data.csv", index=False)
    print("Data opgeslagen als hbo_kennisbank_data.csv")

ModuleNotFoundError: No module named 'tqdm'

### Prompt 3

Uit de vorige prompt kregen wij abstracts die ook de details van het pagina pakte in het abstract, samen met de titel. Dit volgende prompt heeft als doel om dit te voorkomen. Ook proberen wij nog andere details te pakken, zoals het volledige datum, categorie en publicatiesoort. 

In deze prompt geven wij een duidelijke voorbeeld voor het model om het code aan te kunnen passen.

Methodes:
- Aanpassingen op bestaande code: de code van het vorige stap wordt hier aangepast om extra details mee te kunnen nemen.
- Example-based prompting: er is een duidelijke voorbeeld gegeven van het HTML structuur waar het webscraper details moet pakken. Dit is handmatig gepakt uit het HTML zodat het generatieve model rekening mee kan houden.

### Antwoord 3

### Verwerken

Hoewel het abstract goed gepakt wordt met deze code, kan het de details niet goed pakken. Hier heb ik een aantal verschillende prompts voor gebruikt, maar geen een heeft goed gewerkt. In prompt 4 zullen we dan vooral kijken naar het uitbreiden van deze bestaande code die de abstract netjes heeft gehaald om die details ook mee te kunnen nemen.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
from urllib.parse import urljoin

BASE_URL = "https://hbo-kennisbank.nl"
SEARCH_URL = BASE_URL + "/searchresult?has-link=yes&p={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def safe_text(el):
    return el.get_text(strip=True) if el else None


# -----------------------------
# Links ophalen
# -----------------------------
def get_result_links(page):
    url = SEARCH_URL.format(page)
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    links = []
    for a in soup.select("a"):
        href = a.get("href")
        if href and "/details/" in href:
            links.append(urljoin(BASE_URL, href))

    return list(set(links))


# -----------------------------
# Detailpagina scrapen
# -----------------------------
def scrape_detail(url):
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        # Titel
        title = safe_text(soup.find("h1"))

        # -----------------------------
        # Samenvatting (jouw XPath vertaald)
        # /html/body/div[5]/div[3]/div[1]/p
        # -----------------------------
        abstract = None
        try:
            abstract = safe_text(
                soup.select_one("body > div:nth-of-type(5) > div:nth-of-type(3) > div:nth-of-type(1) > p")
            )
        except:
            pass

        # -----------------------------
        # Metadata tabel parsen
        # -----------------------------
        date = None
        department = None
        pub_type = None

        rows = soup.select("body > div:nth-of-type(5) table tbody tr")

        for row in rows:
            cols = row.find_all("td")
            if len(cols) < 2:
                continue

            key = cols[0].get_text(strip=True).lower()
            value_cell = cols[1]

            if "datum" in key:
                date = safe_text(value_cell)

            elif "afdeling" in key:
                # knop zit binnen td
                btn = value_cell.find("button")
                department = safe_text(btn) if btn else safe_text(value_cell)

            elif "type" in key:
                btn = value_cell.find("button")
                pub_type = safe_text(btn) if btn else safe_text(value_cell)

        return {
            "title": title,
            "abstract": abstract,
            "date": date,
            "department": department,
            "type": pub_type,
            "url": url
        }

    except Exception as e:
        print(f"Error bij {url}: {e}")
        return None


# -----------------------------
# Hoofd scraper
# -----------------------------
def scrape_hbo_kennisbank(max_pages=5, delay=1):
    all_links = []

    print("Links verzamelen...")
    for page in range(1, max_pages + 1):
        links = get_result_links(page)
        print(f"Pagina {page}: {len(links)} links")
        all_links.extend(links)
        time.sleep(delay)

    all_links = list(set(all_links))
    print(f"Totaal unieke links: {len(all_links)}")

    data = []
    print("Detailpagina's scrapen...")

    for link in tqdm(all_links):
        item = scrape_detail(link)
        if item and item["title"]:
            data.append(item)
        time.sleep(delay)

    df = pd.DataFrame(data)
    return df


# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    df = scrape_hbo_kennisbank(max_pages=5, delay=1)

    print(df.head())

    df.to_csv("hbo_kennisbank_clean.csv", index=False)
    print("Opgeslagen als hbo_kennisbank_clean.csv")

Links verzamelen...
Pagina 1: 10 links
Pagina 2: 10 links
Totaal unieke links: 20
Detailpagina's scrapen...


100%|██████████| 20/20 [00:24<00:00,  1.20s/it]

                                               title  \
0                             Spelen met de gedachte   
1                       Klimaatadaptief herbestemmen   
2  Een betere mentale gezondheid begint niet in d...   
3  Understanding shelters as children’s places : ...   
4  Occupational Therapy at Home E-Rehabilitation ...   

                                            abstract  date department  type  \
0  Dit praktijkgerichte, kwalitatieve onderzoek r...  None       None  None   
1  Dit onderzoek heet ‘Klimaatadaptief herbestemm...  None       None  None   
2  Opiniestuk in de Volkskrant. Een pleidooi dat ...  None       None  None   
3  The number of families with children experienc...  None       None  None   
4  PURPOSE: To evaluate the feasibility of the Oc...  None       None  None   

  language                                                url  
0     None  https://hbo-kennisbank.nl/details/sharekit_hu:...  
1     None  https://hbo-kennisbank.nl/details/sharekit_han..

In [ ]:
import pandas as pd

df = pd.read_csv("hbo_kennisbank_data.csv")


df['abstract']

0     Verpleegkundig perspectief op persoonsgerichte...
1     Op koers blijven als zijinstromerkwaliteitsvol...
2     Jouw ervaringen en ervaringskennis: werkbladen...
3     Urban Upcycling: Van experiment naar haalbaar ...
4     Ervaringskennis in de organisatie: werkbladLie...
                            ...                        
73    Handelingsperspectief klimaatadaptatie landbou...
74    Meer dan fietsen voor het klimaat: naar effect...
75    Wijkscan gezonde leefomgevingbewegen en ontmoe...
76    Academic stress and mental fatigue predict sub...
77    De misinformatie SafariRuben Smit (Student);Ma...
Name: abstract, Length: 78, dtype: object

### Prompt 4

Zoals in de verwerking van prompt 3 is onderbouwd, hebben wij succesvol de abstract clean kunnen pakken uit de HBO Kennisbank. Het doel van deze volgende prompt is vooral om verdere details (specifiek de afdeling, datum, type en taal) te pakken. 

In prompt 4 gebruiken we weer een aantal technieken om de webscraper te bouwen. 

- Verwachtingen vastgelegd: extra details toegevoegd aan de webscraper en geprompt om het 'copy-paste klaar' te maken zodat het alle codeblokken geeft.
- Specification prompting: we gebruiken specifieke waardes die in de HTML voorkomen zodat de webscraper dit kan vinden. Hiervoor wordt er verwezen naar specifieke elementen in het code. Ook duiden we aan wat met deze waardes kan worden gedaan. 
- Debug prompting: Het doel van het prompt is om een eerdere fout te verbeteren.

### Antwoord 4

### Verwerkte resultaat

In dit resultaat is het prompting niet gelukt, ook na een heleboel verschillende prompts te gebruiken. Uiteindelijk is het belangrijkst voor ons doel om de titel en de abstract te kunnen pakken van alle gescrapete publicaties, wat wel goed gelukt is met het resultaat van prompt 3. De antwoord van deze prompt is uiteindelijk niet toegepast in het eindresultaat.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
from urllib.parse import urljoin

BASE_URL = "https://hbo-kennisbank.nl"
SEARCH_URL = "https://hbo-kennisbank.nl/searchresult?page={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def safe_text(el):
    return el.get_text(strip=True) if el else None


# -----------------------------
# Links ophalen
# -----------------------------
def get_result_links(page):
    url = SEARCH_URL.format(page)
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    links = []
    for a in soup.select("a"):
        href = a.get("href")
        if href and "/details/" in href:
            links.append(urljoin(BASE_URL, href))

    return list(set(links))


# -----------------------------
# Detailpagina scrapen
# -----------------------------
def scrape_detail(url):
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        # Titel
        title = safe_text(soup.find("h1"))

        # -----------------------------
        # Abstract (samenvatting)
        # -----------------------------
        abstract = None
        try:
            abstract = safe_text(
                soup.select_one("body > div:nth-of-type(5) > div:nth-of-type(3) > div:nth-of-type(1) > p")
            )
        except:
            pass

        # -----------------------------
        # Metadata via data-key
        # -----------------------------
        metadata = {
            "department": None,
            "date": None,
            "type": None,
            "language": None
        }

        rows = soup.select('[data-catalog="details"] tr')

        for row in rows:
            key = row.get("data-key")
            if key in metadata:
                tds = row.find_all("td")
                if len(tds) >= 2:
                    value_cell = tds[1]

                    # Soms zit tekst in button
                    btn = value_cell.find("button")
                    if btn:
                        metadata[key] = safe_text(btn)
                    else:
                        metadata[key] = safe_text(value_cell)

        return {
            "title": title,
            "abstract": abstract,
            "date": metadata["date"],
            "department": metadata["department"],
            "type": metadata["type"],
            "language": metadata["language"],
            "url": url
        }

    except Exception as e:
        print(f"Error bij {url}: {e}")
        return None


# -----------------------------
# Hoofd scraper
# -----------------------------
def scrape_hbo_kennisbank(max_pages=5, delay=1):
    all_links = []

    print("Links verzamelen...")
    for page in range(1, max_pages + 1):
        links = get_result_links(page)
        print(f"Pagina {page}: {len(links)} links")
        all_links.extend(links)
        time.sleep(delay)

    all_links = list(set(all_links))
    print(f"Totaal unieke links: {len(all_links)}")

    data = []
    print("Detailpagina's scrapen...")

    for link in tqdm(all_links):
        item = scrape_detail(link)
        if item and item["title"]:
            data.append(item)
        time.sleep(delay)

    df = pd.DataFrame(data)
    return df


# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    df = scrape_hbo_kennisbank(max_pages=5, delay=1)

    print(df.head())

    df.to_csv("hbo_kennisbank_final.csv", index=False)
    print("Opgeslagen als hbo_kennisbank_final.csv")

Links verzamelen...
Pagina 1: 10 links
Pagina 2: 10 links
Pagina 3: 9 links
Pagina 4: 9 links
Pagina 5: 6 links
Totaal unieke links: 43
Detailpagina's scrapen...


100%|██████████| 43/43 [00:51<00:00,  1.20s/it]

                                               title  \
0                             Spelen met de gedachte   
1  Laadbehoefte in de praktijk: Elektriciteitsvra...   
2  Komen tot collectieve ervaringskennis stimulee...   
3                            Thinking out of the Box   
4                       Klimaatadaptief herbestemmen   

                                            abstract  date department  type  \
0  Dit praktijkgerichte, kwalitatieve onderzoek r...  None       None  None   
1  In het project “Laadbehoefte in de praktijk” o...  None       None  None   
2  Veel gemeenten en uitvoeringsorganisaties will...  None       None  None   
3  Eco-criticism, emerging in literary studies in...  None       None  None   
4  Dit onderzoek heet ‘Klimaatadaptief herbestemm...  None       None  None   

  language                                                url  
0     None  https://hbo-kennisbank.nl/details/sharekit_hu:...  
1     None  https://hbo-kennisbank.nl/details/amsterdam_pu..